In [2]:
# 02 - Profile raw data

# Purpose: understand the raw data exactly as loaded, before changing anything. Every decision made from these findings is logged in `docs/decision-log.md`.

In [3]:
import pandas as pd

DATA_PATH = "../data/bank-additional-full.csv"
raw_df = pd.read_csv(DATA_PATH, sep=";", dtype=str, keep_default_na=False)
raw_df.shape

(41188, 21)

In [4]:
# Profile step 1: blanks and "unknown" values per column
stripped_df = raw_df.apply(lambda column: column.str.strip())

missing_summary = pd.DataFrame({
    "blank_count": (stripped_df == "").sum(),
    "unknown_count": (stripped_df == "unknown").sum(),
})
missing_summary["total_incomplete"] = (
    missing_summary["blank_count"] + missing_summary["unknown_count"]
)
missing_summary["percent_incomplete"] = (
    missing_summary["total_incomplete"] / len(raw_df) * 100
).round(2)

missing_summary.sort_values("percent_incomplete", ascending=False)

,blank_count,unknown_count,total_incomplete,percent_incomplete
default,0,8597,8597,20.87
education,0,1731,1731,4.20
housing,0,990,990,2.40
loan,0,990,990,2.40
job,0,330,330,0.80
marital,0,80,80,0.19
age,0,0,0,0.00
contact,0,0,0,0.00
month,0,0,0,0.00
day_of_week,0,0,0,0.00


In [5]:
# Profile step 2: do the "unknown" values overlap across columns?
unknown_columns = ["default", "education", "housing", "loan", "job", "marital"]
is_unknown_df = stripped_df[unknown_columns] == "unknown"

# How many rows have 0, 1, 2, ... unknown values?
unknowns_per_row = is_unknown_df.sum(axis=1)
print("Rows by number of unknown values:")
print(unknowns_per_row.value_counts().sort_index())
print()

# Are the housing and loan unknowns the same rows?
pd.crosstab(
    is_unknown_df["housing"],
    is_unknown_df["loan"],
    rownames=["housing unknown"],
    colnames=["loan unknown"],
)

Rows by number of unknown values:
0    30488
1     9034
2     1338
3      306
4       20
5        2
Name: count, dtype: int64



loan unknown,False,True
housing unknown,,
False,40198,0
True,0,990


In [6]:
# Profile step 3: yes rate for "unknown" vs known customers, per column
print("Values in y:")
print(stripped_df["y"].value_counts())
print()

is_yes = stripped_df["y"] == "yes"
print(f"Overall yes rate: {is_yes.mean() * 100:.2f}%")
print()

comparison_rows = []
for column in unknown_columns:
    unknown_mask = is_unknown_df[column]
    comparison_rows.append({
        "column": column,
        "unknown_rows": unknown_mask.sum(),
        "yes_rate_unknown_pct": round(is_yes[unknown_mask].mean() * 100, 2),
        "known_rows": (~unknown_mask).sum(),
        "yes_rate_known_pct": round(is_yes[~unknown_mask].mean() * 100, 2),
    })

pd.DataFrame(comparison_rows).set_index("column")

Values in y:
y
no     36548
yes     4640
Name: count, dtype: int64

Overall yes rate: 11.27%



,unknown_rows,yes_rate_unknown_pct,known_rows,yes_rate_known_pct
column,,,,
default,8597,5.15,32591,12.88
education,1731,14.50,39457,11.12
housing,990,10.81,40198,11.28
loan,990,10.81,40198,11.28
job,330,11.21,40858,11.27
marital,80,15.00,41108,11.26


In [7]:
# Profile step 4: pdays = 999 and poutcome
print("Values in poutcome:")
print(stripped_df["poutcome"].value_counts())
print()

pdays_is_999 = stripped_df["pdays"] == "999"
print(f"Rows with pdays = 999: {pdays_is_999.sum()} ({pdays_is_999.mean() * 100:.2f}%)")
print()

print("pdays = 999 vs poutcome:")
print(pd.crosstab(pdays_is_999, stripped_df["poutcome"],
                  rownames=["pdays is 999"], colnames=["poutcome"]))
print()

print("pdays = 999 vs previous = 0:")
print(pd.crosstab(pdays_is_999, stripped_df["previous"] == "0",
                  rownames=["pdays is 999"], colnames=["previous is 0"]))
print()

real_pdays = pd.to_numeric(stripped_df.loc[~pdays_is_999, "pdays"], errors="coerce")
print("pdays excluding 999 (should be real day counts):")
print(real_pdays.describe())
print(f"Values that failed conversion: {real_pdays.isna().sum()}")

Values in poutcome:
poutcome
nonexistent    35563
failure         4252
success         1373
Name: count, dtype: int64

Rows with pdays = 999: 39673 (96.32%)

pdays = 999 vs poutcome:
poutcome      failure  nonexistent  success
pdays is 999                               
False             142            0     1373
True             4110        35563        0

pdays = 999 vs previous = 0:
previous is 0  False  True 
pdays is 999               
False           1515      0
True            4110  35563

pdays excluding 999 (should be real day counts):
count    1515.000000
mean        6.014521
std         3.824906
min         0.000000
25%         3.000000
50%         6.000000
75%         7.000000
max        27.000000
Name: pdays, dtype: float64
Values that failed conversion: 0


In [8]:
# Profile step 5: numeric columns and duplicate rows
numeric_columns = [
    "age", "duration", "campaign", "pdays", "previous",
    "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed",
]
numeric_df = stripped_df[numeric_columns].apply(pd.to_numeric, errors="coerce")

print("Failed conversions per numeric column:")
print(numeric_df.isna().sum())
print()

extra_copies = stripped_df.duplicated().sum()
rows_in_repeated_groups = stripped_df.duplicated(keep=False).sum()
print(f"Extra duplicate copies: {extra_copies}")
print(f"Rows belonging to a repeated group (originals included): {rows_in_repeated_groups}")
print()

print("Numeric columns, min / median / max:")
numeric_df.describe().T[["min", "50%", "max"]]

Failed conversions per numeric column:
age               0
duration          0
campaign          0
pdays             0
previous          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
dtype: int64

Extra duplicate copies: 12
Rows belonging to a repeated group (originals included): 24

Numeric columns, min / median / max:


,min,50%,max
age,17.000,38.000,98.000
duration,0.000,180.000,4918.000
campaign,1.000,2.000,56.000
pdays,0.000,999.000,999.000
previous,0.000,0.000,7.000
emp.var.rate,-3.400,1.100,1.400
cons.price.idx,92.201,93.749,94.767
cons.conf.idx,-50.800,-41.800,-26.900
euribor3m,0.634,4.857,5.045
nr.employed,4963.600,5191.000,5228.100


In [9]:
# Profile step 6: duplicates and unusual values
is_yes = stripped_df["y"] == "yes"

# 1. The duplicate pairs (all 21 columns match within a pair; showing key columns)
key_columns = ["age", "job", "month", "day_of_week", "duration", "campaign", "y"]
duplicate_pairs = stripped_df[stripped_df.duplicated(keep=False)]
print("Duplicate pairs:")
print(duplicate_pairs.sort_values(key_columns)[key_columns].to_string())
print()

# 2. Worst-case effect of the duplicates on the yes rate
deduped_df = stripped_df.drop_duplicates()
print(f"Rows: {len(stripped_df)} -> {len(deduped_df)} after removing extra copies")
print(f"Yes rate with duplicates:       {is_yes.mean() * 100:.3f}%")
print(f"Yes rate without extra copies:  {(deduped_df['y'] == 'yes').mean() * 100:.3f}%")
print()

# 3. Unusual values: how many rows, and do they behave differently?
def describe_group(label, mask):
    print(f"{label}: {mask.sum()} rows, yes rate {is_yes[mask].mean() * 100:.2f}%")

describe_group("age under 18", numeric_df["age"] < 18)
describe_group("duration = 0", numeric_df["duration"] == 0)
describe_group("campaign above 10", numeric_df["campaign"] > 10)
describe_group("campaign above 30", numeric_df["campaign"] > 30)
print()

# 4. Do the economic indicators describe the period, not the customer?
economic_columns = ["emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]
print("Distinct values per economic indicator:")
print(numeric_df[economic_columns].nunique())

Duplicate pairs:
      age          job month day_of_week duration campaign    y
28476  24     services   apr         tue      114        1   no
28477  24     services   apr         tue      114        1   no
14155  27   technician   jul         mon      331        2   no
14234  27   technician   jul         mon      331        2   no
18464  32   technician   jul         thu      128        1   no
18465  32   technician   jul         thu      128        1   no
32505  35       admin.   may         fri      348        4   no
32516  35       admin.   may         fri      348        4   no
12260  36      retired   jul         thu       88        1   no
12261  36      retired   jul         thu       88        1   no
25183  39       admin.   nov         tue      123        2   no
25217  39       admin.   nov         tue      123        2   no
1265   39  blue-collar   may         thu      124        1   no
1266   39  blue-collar   may         thu      124        1   no
20531  41   technician 